<a href="https://colab.research.google.com/github/prithwis/parashar21/blob/main/P21_300YogDarshan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/parashar21/main/images/khona21-v0.png) <br>


---



[Prithwis Mukerjee](http://www.yantrajaal.com) ||
Website - [Parashar21](https://parashar21.blogspot.com)<br>

# Khona21 — Interpretive Jyotiṣa

**Khona21** is an experimental AI-assisted Jyotiṣa interpretation platform, developed as a companion to **Parashar21**, which performs the astronomical calculations underlying traditional Indian astrology.

Named after the legendary Bengali astrologer Khona, the platform combines precomputed Rashi and Navamsha charts, Vimshottari Dasha schedules and planetary transits (Gochar) with a curated corpus of Jyotiṣa literature, currently comprising works by **James Braha** and **Harihar Majumdar**.

Rather than relying on hard-coded interpretive rules, Khona21 employs **source-grounded, multipass LLM reasoning** to examine planetary configurations, yogas, cancellations, exceptions and conflicting interpretations. An initial assessment establishes the horoscope's principal characteristics, while subsequent consultations investigate specific questions using the original reference texts and relevant astrological evidence.

This notebook, **k21_KhonaPlus**, implements the consultation workflow, integrating deterministic calculations from Parashar21 with the interpretive capabilities of modern Large Language Models.

**Khona21 explores the convergence of traditional Jyotiṣa knowledge and modern artificial intelligence, combining computational precision with source-grounded reasoning to produce systematic, transparent and auditable astrological interpretations.**

In [1]:
from datetime import datetime
import pytz
print("\033[1m"+'জন্ম-মৃত্যু-বিবাহ, তিন না জানেন বরাহ।'+"\033[0m")
print(datetime.now(pytz.timezone('Asia/Calcutta')))

জন্ম-মৃত্যু-বিবাহ, তিন না জানেন বরাহ।
2026-11-04 18:49:32.632627+05:30


#Setup, Installs, Imports

In [2]:
!pip -q install pypandoc
!pip -q install python-docx
from IPython.display import display, Markdown


!wget -O k21core.py -q https://raw.githubusercontent.com/prithwis/parashar21/main/k21utils/k21core.py
!wget -O k21fileops.py -q https://raw.githubusercontent.com/prithwis/parashar21/main/k21utils/k21fileops.py
from k21core import *
from k21fileops import *

In [3]:
# ------------------------------------------------
# required for multiple imports of the same module
# ------------------------------------------------
#import k21core
#import importlib

#importlib.reload(k21core)
from k21core import *

In [ ]:
# ------------------------------------------------
# Use Claude
# ------------------------------------------------
#!pip install -q anthropic

#provider = "claude"
#model = "claude-haiku-5-5"


In [4]:
# ------------------------------------------------
# Use OpenAI
# ------------------------------------------------
provider = "openai"

#model = "gpt-5.6-luna"
#model = "gpt-6-luna"
model = "gpt-6.1-sol"
#model = "gpt-6-astra"

In [ ]:
# ------------------------------------------------
# Set Native ID
# ------------------------------------------------
Native_id = "TopGun"

###Access Control
This notebook accesses Google Drive to access files and uses the OpenAI API. Hence it can be used ONLY from the userid of the creator. Others will have to create their own access paths

In [5]:
# ------------------------------------------------
# Authenticate wit LLM provider using API keys stored as Colab Secret
# ------------------------------------------------
client = authenticateLLM(provider=provider)

OPENAI | User: Calcutta | Email: prithwis@yantrajaal.com | ID: user-sNAljKNmBf2igtAV2kUuqhoF


In [6]:
# ------------------------------------------------
# Locate necessary file and set up paths to the same
# ------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')
#ls /content/drive/MyDrive/Prithwis/Parashar21/K21

K21_HOME = "/content/drive/MyDrive/Prithwis/Parashar21/K21"

# ------------------------------------------------
# The Corpus Files, Braha and Harihar, are available
# in GitHub Repo and can be downloaded with wget
# ------------------------------------------------
#BRAHA   = f"{K21_HOME}/Corpus/BR_Curated_22.txt"
#HARIHAR = f"{K21_HOME}/Corpus/HM_Clean_07.txt"
#RAMAN = f"{K21_HOME}/Corpus/Raman_08.md"
YOGS = f"{K21_HOME}/Corpus/300_Astrology_Yogas.md"



Mounted at /content/drive


## Load Files

In [ ]:
#!wget -O braha.txt -q "https://raw.githubusercontent.com/prithwis/parashar21/main/Corpus Texts/Text_Extracts/BR_Curated_22.txt"
#!wget -O harihar.txt -q "https://raw.githubusercontent.com/prithwis/parashar21/main/Corpus Texts/Text_Extracts/HM_Clean_07.txt"
#!wget -O role.txt -q https://raw.githubusercontent.com/prithwis/parashar21/main/RoleFiles/khona261029A.txt

#BRAHA   = "/content/braha.txt"
#HARIHAR = "/content/harihar.txt"
#ROLE    = "/content/role.txt"

In [7]:
#
# Following files from Parashar21 need to be uploaded here
#

import zipfile
from google.colab import files

# Upload ZIP file
uploaded = files.upload()
zipName = next(iter(uploaded))

# Extract all files into current directory
with zipfile.ZipFile(zipName, 'r') as zf:
    zf.extractall()

print("Extracted:", zipName)

CHART = "/content/Chart.txt"
NAVAMSA = "/content/Navamsa_Chart.txt"
DASHA = "/content/Dasha.txt"
GOCHAR = "/content/Gochar_Chart.txt"
GOCHAR_EVENTS = "/content/Transit.txt"

Saving Kompute_261104_132217.zip to Kompute_261104_132217.zip
Extracted: Kompute_261104_132217.zip


# LLM Access




In [8]:
# ------------------------------------------------
# Define the ROLE
# ------------------------------------------------
ROLE1 = f"{K21_HOME}/Roles/ramanyog261104A.txt"
raman = readText(ROLE1)
#ROLE2 = f"{K21_HOME}/Roles/khona261104A.txt"
#khona21 = readText(ROLE2)

/content/drive/MyDrive/Prithwis/Parashar21/K21/Roles/ramanyog261104A.txt | 4,533 characters


## 300 YOGS

In [9]:
# ------------------------------------------------
# Set the Context
# ------------------------------------------------
#BaseCONTEXT = buildContext(BRAHA, HARIHAR, CHART, NAVAMSA, DASHA)


CONTEXT = getContext(
    RAMANYOG=YOGS,
    CHART=CHART,
    NAVAMSA=NAVAMSA
)


In [10]:
len(CONTEXT)

192943

In [11]:
import tiktoken

enc = tiktoken.get_encoding("o200k_base")
n_tokens = len(enc.encode(CONTEXT))
print(f"Context tokens: {n_tokens:,}")

Context tokens: 42,141


In [13]:
YOG_INSTRUCTION = """
Examine all yoga definitions in YOGS against CHART and NAVAMSA
Identify and verify applicable yogas, documenting their formation
conditions, chart evidence and stated outcomes.
Report uncertain formations separately.
"""

In [14]:
# ------------------------------------------------
# Assemble the yog query
# Call the LLM
# ------------------------------------------------

QUERY = buildQuery(CONTEXT, YOG_INSTRUCTION)

RamanYog = callLLM(
    client=client,
    role=raman,
    query=QUERY,
    model=model,
    provider=provider,
    effort="medium"
)

OPENAI | gpt-6.1-sol | Effort: medium | In: 43,083 | Cached: 0 | Out: 12,207 | Reasoning: 7,250 | Time: 304.4s


In [15]:
# ------------------------------------------------
# Save RamanYog as markdown for machine use
# ------------------------------------------------
MD_FILE = saveAnswer(model, RamanYog)
display(Markdown(RamanYog))

Saved: /content/K21_261104_190229_gpt-6.1-sol.md


# RAMANYOG Verification Report

Source-entry numbers below follow the order of the **300 supplied yoga definitions**, preserving repeated names. D1 denotes CHART; D9 denotes NAVAMSA. Supplied divisional signs and houses are used without recalculating them from the repeated longitude fields.

Unqualified “benefic” and “malefic” wording is not automatically equated with CHART’s **functional** classifications. Where that distinction changes verification, the formation is reported separately.

**Stated outcomes are source attributions, not established personal characteristics, medical findings or guaranteed events.**

## 1. A. VERIFIED YOGAS

### 1.1 Sunapha Yoga — source entry 2

- **Formation:** “Any planets, except Sun, in the second house from the Moon.”
- **Chart evidence:** Moon occupies Sagittarius. Capricorn, second from Sagittarius, contains Jupiter and Saturn.
- **Stated outcomes:** Self-earned property, substantial wealth, intelligence, sound decisions and ruler-like reputation.
- **Qualifications:** Neither strength nor freedom from affliction is required by this entry. Jupiter’s debilitation does not invalidate its stated formation.

### 1.2 Sakata Yoga — source entry 12

- **Formation:** “Moon in 12th, 6th or 8th house from Jupiter.”
- **Chart evidence:** Jupiter occupies Capricorn; Moon in Sagittarius is twelfth from Jupiter.
- **Stated outcomes:** Loss and subsequent recovery of fortune, hardship, stubbornness and difficulties with relatives.
- **Qualifications:** This is the Moon–Jupiter formation, distinct from the later same-name planetary-distribution entry.

### 1.3 Vesi Yoga — source entry 16

- **Formation:** “Planets other than Moon occupy 2nd position from Sun.”
- **Chart evidence:** Sun occupies Virgo. Mars and Mercury occupy Libra, second from Sun.
- **Stated outcomes:** Fortune, happiness, ethical conduct, fame and aristocratic standing.
- **Qualifications:** Verification does not depend on whether nodes count as planets.

### 1.4 Sasa Yoga — source entry 21

- **Formation:** “Saturn occupies a Kendra of his own house or exaltation sign.”
- **Chart evidence:** Saturn occupies D1 house 7, a kendra, in Capricorn, explicitly identified as its own house.
- **Stated outcomes:** Authority over others, leadership or rulership; also questionable conduct, covetousness and anger.
- **Qualifications:** Saturn’s conjunction with debilitated Jupiter and aspect from Mars are supplied chart facts, but this entry provides no cancellation rule based on them.

### 1.5 Parijatha Yoga — source entry 47

- **Formation:** “Lord of the sign in which the lord of the house occupied by the Ascendant lord, or the lord of Navamasa occupied by the lord of Rasi in which the Ascendant lord is positioned joins a quadrant, a trine or his own exaltation signs.”
- **Chart evidence:** The explicit Navamsa alternative is satisfied:
  1. Ascendant lord Moon occupies Sagittarius.
  2. Sagittarius lord is Jupiter.
  3. Jupiter occupies Aquarius in D9.
  4. Aquarius lord Saturn occupies D1 house 7, a quadrant.
- **Stated outcomes:** King-like standing, happiness particularly in middle and later life, recognition, generosity and adherence to traditions.
- **Qualifications:** Verification rests on the intelligible Navamsa alternative; it does not require repairing the less complete first alternative.

### 1.6 Sareera Soukhya Yoga — source entry 108

- **Formation:** “The lord of Lagna, Jupiter or Venus should occupy a quadrant.”
- **Chart evidence:** Jupiter occupies D1 house 7, satisfying one expressly permitted alternative.
- **Stated outcomes:** Longevity, freedom from major disease, wealth, prosperity and interest in politics.
- **Qualifications:** Jupiter is debilitated. The formation clause does not require dignity or strength; the attributed health and longevity outcomes are not verified by formation alone.

### 1.7 Swaveeryaddhana Yoga — source entry 132

- **Formation:** “The 2nd lord should occupy a quadrant or trine from the 1st lord or the 2nd lord being a bene fic should be in either a deep exaltation or conjunction with an exalted planet.”
- **Chart evidence:** First lord Moon occupies Sagittarius; second lord Sun occupies Virgo, tenth from Moon—a quadrant.
- **Stated outcomes:** Wealth through personal effort, hard work and self-created opportunities or enterprise.
- **Qualifications:** The first alternative is sufficient. No benefic classification or exaltation assumption is needed.

### 1.8 Yuddha Praveena Yoga — source entry 183

- **Formation:** “The Lord of the navamsa joined by the planet that owns the navamsa in which the third Lord is placed, joins its own vargas.”
- **Chart evidence:**
  1. D1 third lord Mercury occupies Capricorn in D9; its lord is Saturn.
  2. Saturn also occupies Capricorn in D9; that Navamsa’s lord is again Saturn.
  3. Saturn occupies its own Capricorn in both supplied D1 and D9.
- **Stated outcomes:** Sound judgement, planning ability, strategic intelligence and possible proficiency in war strategy.
- **Qualifications:** Own-varga evidence is established for the two supplied divisions only; no additional divisional dignity is inferred.

### 1.9 Bandhu Pujya Yoga — source entry 194

- **Formation:** “The fourth house or the fourth Lord has the association or aspect of Jupiter.”
- **Chart evidence:** Fourth lord Venus receives Jupiter’s explicitly supplied aspect.
- **Stated outcomes:** Affection and respect from relatives and associates, empathy and trustworthiness; this entry says the person is more loved than revered.
- **Qualifications:** This is distinct from source entry 193, which has additional conditions.

### 1.10 Bandhubhisthyaktha Yoga — source entry 195

- **Formation:** “The fourth Lord is associated with malefics or occupies evil shashtiamsas or joins inimical or debilitation signs.”
- **Chart evidence:** Fourth lord Venus is explicitly debilitated in Virgo.
- **Stated outcomes:** Misunderstanding, displeasure or abandonment by relatives and associates.
- **Qualifications:** The debilitation alternative is independently sufficient; no Shashtiamsa or malefic classification is assumed.

### 1.11 Nishkapata Yoga — source entry 205

- **Formation:** “The fourth house is occupied by a benefic, or a planet in exaltation, friendly or own house, or the fourth house is a benefic sign.”
- **Chart evidence:** Mercury occupies D1 house 4 and is explicitly recorded as being in a friendly house.
- **Stated outcomes:** Sincerity, a loving disposition, dislike of hypocrisy and preference for openness.
- **Qualifications:** Verification uses the friendly-house alternative, not a natural-benefic classification of Mercury.

### 1.12 Ekaputra Yoga — source entry 225

- **Formation:** “Lord of the fifth house joins a kendra or trikona.”
- **Chart evidence:** Fifth lord Mars occupies D1 house 4, a kendra.
- **Stated outcomes:** One child, described primarily as a son; the source also discusses inability or unwillingness to have additional children.
- **Qualifications:** Formation does not establish the actual number, sex or survival of children.

### 1.13 Satkalatra Yoga — source entry 239

- **Formation:** “The Lord of the seventh house or Venus is joined or is aspected by Jupiter or Mercury.”
- **Chart evidence:** Seventh lord Saturn is explicitly conjunct Jupiter in D1 house 7.
- **Stated outcomes:** A noble, disciplined, virtuous and attached spouse.
- **Qualifications:** Jupiter’s debilitation is established, but this entry imposes no strength or dignity requirement.

### 1.14 Raja Yoga — source entry 250

- **Formation:** “The Lord of the sign, a planet is debilitated in, or the planet, that would be exalted there, is in a kendra from the Moon or Lagna.”
- **Chart evidence:** Two independent matches occur:
  - Jupiter is debilitated in Capricorn; Capricorn lord Saturn occupies house 7 from Lagna.
  - Venus is debilitated in Virgo; Virgo lord Mercury occupies house 4 from Lagna.
- **Stated outcomes:** Ruler-like influence, respect, power and substantial wealth.
- **Qualifications:** Both matches belong to one source entry and count once. This verifies the stated relationship, not an independently assumed restoration of planetary strength or elimination of other formations.

## 2. B. CONDITIONAL YOGAS

### 2.1 Vasumathi Yoga — source entry 9

- **Formation:** “Benefics occupy the upachayas 3, 6, 10, or 11 either from the ascendant or from the Moon.”
- **Possible match:** Functionally benefic Moon occupies house 6 from Lagna; functionally benefic Mars occupies house 11 from Moon. Venus occupies house 3 from Lagna and Mercury house 10 from Moon.
- **Unresolved condition:** The source does not specify natural versus functional benefics, whether one qualifying planet suffices, or whether a common reference point is required.
- **Stated outcomes:** Wealth, prosperity, independence, social respect and generosity.

### 2.2 Amala Yoga — source entry 13

- **Formation:** “10th from Moon or Lagna should be occupied by any benefic planet”
- **Possible match:** Virgo, tenth from Moon, contains Venus and Sun.
- **Unresolved condition:** Venus is not explicitly classified as a benefic in the supplied D1 data. Verification requires an unstated natural-benefic convention.
- **Stated outcomes:** Lasting reputation, good character, prosperity and health.

### 2.3 Parvata Yoga — source entry 14

- **Formation:** “6th or 8th houses should be either unoccupied or occupied by benefic planets.”
- **Possible match:** House 6 contains Moon, explicitly functionally benefic.
- **Unresolved condition:** Whether “benefic” includes this functional classification. The literal “or” permits the sixth-house alternative independently; it is not silently changed to require both houses.
- **Stated outcomes:** Wealth, prosperity, charity, humour, passion and local leadership.

### 2.4 Vasi Yoga — source entry 17

- **Formation:** “Planets other than Moon occupy 12th position from Sun.”
- **Possible match:** Rahu occupies Leo, twelfth from Sun in Virgo.
- **Unresolved condition:** Whether the entry includes lunar nodes among “planets.”
- **Stated outcomes:** Happiness, prosperity, liberality and favour from ruling classes.

### 2.5 Obhayachari Yoga — source entry 18

- **Formation:** “Planets other than Moon are on either side of the Sun.”
- **Possible match:** Rahu is immediately before Sun’s sign; Mars and Mercury are immediately after it.
- **Unresolved condition:** The preceding-side occupancy depends entirely on including Rahu.
- **Stated outcomes:** Eloquence, proportionate physique, happiness, wealth and fame.

### 2.6 Brahma Yoga — source entry 63

- **Formation:** “Jupiter and Venus are in Kendras respectively from the lords of the 9th and 11th, and Mercury is in a similar position from the lord of either Lagna or the 10th.”
- **Possible match:** Ninth lord is Jupiter itself; eleventh lord is Venus itself. Mercury is conjunct tenth lord Mars.
- **Unresolved condition:** The first two relationships require treating each planet’s position relative to itself as satisfying “in Kendras.” That intended self-reference is not established.
- **Stated outcomes:** Luxury, learning, respect from learned people, longevity and charity.

### 2.7 Ardha Chandra Yoga — source entry 79

- **Formation:** “All planets occupy the seven houses beginning from a Panapara or Apoklima.”
- **Possible match:** All supplied planets and nodes lie within houses 2–8.
- **Unresolved condition:** Whether confinement within a seven-house span suffices, rather than occupation of each house, and which starting houses the undefined Panapara/Apoklima terms permit.
- **Stated outcomes:** Pleasing appearance and sustained happiness.

### 2.8 Damni Yoga — source entry 92

- **Formation:** “All planets must occupy any six signs.”
- **Possible match:** Including Rahu and Ketu, the chart occupies six signs: Leo, Virgo, Libra, Sagittarius, Capricorn and Aquarius.
- **Unresolved condition:** Whether “all planets” includes the nodes.
- **Stated outcomes:** Wealth, charity, helping others and protection of cattle.

### 2.9 Kedara Yoga — source entry 94

- **Formation:** “All planets must occupy any four signs.”
- **Possible match:** The seven non-node planets occupy exactly four signs: Virgo, Libra, Sagittarius and Capricorn.
- **Unresolved condition:** Whether nodes are excluded from this entry’s planetary set.
- **Stated outcomes:** Agricultural livelihood, assistance to others and limited comprehension.

### 2.10 Sarpa Yoga — source entry 102

- **Formation:** “All the malefics occupy kendras.”
- **Possible match:** Every planet explicitly labelled a D1 functional malefic—Mercury, Jupiter and Saturn—occupies house 4 or 7.
- **Unresolved condition:** Whether the source intends functional malefics rather than a different natural-malefic set.
- **Stated outcomes:** Misery, lack of enjoyment, duplicity and poor judgement.

### 2.11 Sada Sanchara Yoga — source entry 117

- **Formation:** “The lord of either the Lagna or the sign occupied by Lagna lord must be a movable sign.”
- **Possible match:** Moon occupies Sagittarius; its sign lord Jupiter occupies movable Capricorn.
- **Unresolved condition:** The wording makes a planet “be a movable sign.” A match requires interpreting this as *occupying* a movable sign.
- **Stated outcomes:** Wandering, travel, distraction and interest in unfamiliar places and cultures.

### 2.12 Matrumooladdhana Yoga — source entry 138

- **Formation:** “The lord of the 2nd joins the 4th lord or is aspected by him the above yoga.”
- **Possible match:** Second lord Sun is conjunct fourth lord Venus.
- **Unresolved condition:** The trailing “the above yoga” is an unresolved cross-reference; it may imply omitted conditions.
- **Stated outcomes:** Financial support or wealth through the mother.

### 2.13 Daridra Yoga — source entry 150

- **Formation:** “The lord of the 5th joins with the lord of 6th, 8th or 12th without beneficial aspects or conjunctions.”
- **Possible match:** Fifth lord Mars is conjunct twelfth lord Mercury and receives Saturn’s aspect. Mercury and Saturn are explicitly functional malefics.
- **Unresolved condition:** Absence of beneficial contact is supported only under the supplied functional classifications; the source does not specify that framework.
- **Stated outcomes:** Severe financial difficulty, inability to accumulate wealth and poverty.

### 2.14 Bhratruvriddhi Yoga — source entry 177

- **Formation:** “The third Lord, or Mars, or the third house are joined or aspected by benefics or strong.”
- **Possible match:** Third lord Mercury is conjunct functionally benefic Mars.
- **Unresolved condition:** Whether a functional benefic satisfies the unqualified term “benefics.” No independent strength assessment is supplied.
- **Stated outcomes:** Happiness through siblings, strong family bonds and siblings’ prosperity.

### 2.15 Kapata Yoga — source entry 202

- **Formation:** “The fourth house is joined by a malefic and the fourth Lord is associated with or aspected by malefics or is hemmed in between malefic.”
- **Possible match:** House 4 contains functionally malefic Mercury. Fourth lord Venus in Virgo is flanked by Rahu in Leo and Mercury in Libra.
- **Unresolved condition:** Completion requires an unstated malefic classification for Rahu, or an alternative classification establishing Sun—Venus’s conjunction partner—as malefic.
- **Stated outcomes:** Diplomacy, hypocrisy, inconsistency between preaching and practice, and possible superiority toward others.

### 2.16 Theevrabuddhi Yoga — source entry 232

- **Formation:** “The Lord of the navamsa, in which the Lord of the fifth is placed, is aspected by benefics, and the fifth Lord himself is a benefic.”
- **Possible match:** Fifth lord Mars is functionally benefic. Its D9 sign is Aquarius, whose lord Saturn receives Mars’s D1 aspect.
- **Unresolved condition:** Both benefic requirements are met using Mars’s functional classification, but the source does not establish that this is the intended classification.
- **Stated outcomes:** Precocious intelligence or genius, with recognition dependent on circumstances.

## 3. C. UNVERIFIABLE YOGAS

The following candidates lack essential data or sufficiently intelligible wording.

1. **Vanchana Chora Bheethi Yoga — entry 11:** The alternative involving Gulika’s association with kendra and trine lords cannot be checked; Gulika is not supplied.

2. **Ruchaka Yoga — entry 22:** Mars occupies a kendra, but its required exaltation is not established. Its Libra placement is not its own sign under the supplied lordships.

3. **Bhadra Yoga — entry 23:** Mercury occupies a kendra, but “the native’s own exaltation sign” is defective wording. No intelligible exaltation criterion is supplied for verification.

4. **Sankha Yoga — entry 44:** Fifth lord Mars and sixth lord Jupiter occupy mutual kendras, but the required powerful disposition of ascendant lord Moon is not supplied.

5. **Mridanga Yoga — entry 46:** Mars’s D9 dispositor Saturn occupies a D1 kendra and a supplied friendly sign, but the necessary exalted starting planet and strong ascendant lord are not established.

6. **Jaya Yoga — entry 58:** Sixth lord Jupiter is debilitated; tenth lord Mars’s required deep exaltation is not established.

7. **Garuda Yoga — entry 66:** Moon’s D9 lord is Moon. Its required exaltation, daytime birth and waxing phase are not established.

8. **Krisanga Yoga — entry 112:** No dry-sign or dry-planet classification is supplied. Elemental classifications cannot substitute for dryness.

9. **Swaveeryaddhana Yoga — entry 131:** The dispositor chain reaches Jupiter, which is fifth from second lord Sun. Jupiter’s required strength is not supplied; debilitation alone is not converted into a complete strength assessment.

10. **Netranasa Yoga — entry 164:** The independent Neechamsa alternative cannot be established from the supplied D9 dignity information.

11. **Durmukha Yoga — entry 169:** Gulika is absent, and the alternative requires unprovided unfriendly/debilitated Navamsa and malefic-association information.

12. **Bhojana Soukhya Yoga — entry 170:** Second lord Sun receives Jupiter’s aspect, but Sun’s required strength and Vaiseshikamsa are unavailable.

13. **Parannabhojana Yoga — entry 172:** Sun receives the aspect of debilitated Jupiter, but the necessary debilitation or unfriendly-Navamsa status of Sun is not established.

14. **Sarpaganda Yoga — entry 174:** Rahu occupies house 2; Mandi’s required conjunction is uncheckable.

15. **Vakchalana Yoga — entry 175:** The second lord’s malefic classification and cruel Navamsa are not supplied. The listed second-house contacts alone are insufficient.

16. **VishapraYoga Yoga — entry 176:** House 2 contains Rahu and receives Ketu’s aspect; second lord Sun receives Jupiter’s aspect. Required malefic classifications and Sun’s cruel Navamsa remain unestablished.

17. **Parakrama Yoga — entry 182:** Third lord Mercury joins functionally benefic Mars, but Mercury’s benefic Navamsa and Mars’s benefic-sign status are undefined.

18. **Yuddhatpoorvadridhachitta Yoga — entry 184:** Third lord Mercury occupies movable Libra and movable Capricorn Navamsa, but its required exaltation and the intended malefic classification of its conjunction partner are not established.

19. **Satkathadisravana Yoga — entry 186:** House 3 receives Jupiter’s aspect, but the required benefic-sign, benefic-planet and benefic-amsa classifications are not supplied.

20. **Nishkapata Yoga — entry 206:** The potentially independent Parvata/Uttamamsa alternative cannot be assessed; those specialised amsas are absent.

21. **Matru Sneha Yoga — entry 208:** First lord Moon and fourth lord Venus are distinct. Their required natural or temporal friendship is not supplied, and no independently established qualifying benefic-aspect alternative resolves the entry.

22. **Anapathya Yoga — entry 211:** Weakness of all four required factors—Jupiter and the first, seventh and fifth lords—is not established. Dignity labels are not substituted for a complete strength assessment.

23. **Kalanirdesat Putranasa Yoga — entry 230:** “Malefics are Jupiter and Lagna” does not specify an intelligible positional or relational formation.

24. **Thrikalagnana Yoga — entry 234:** Jupiter receives an aspect from functionally benefic Mars, but the required Mrudwamsa/Gopuramsa information is missing.

25. **Bhagya Yoga — entry 241:** Venus occupies house 3 and aspects house 9, but its required strong-benefic status is not established.

26. **Raja Yoga — entry 246:** Debilitated Jupiter occupies a kendra, but “bright rays” and the intended “favourable positions” criterion are not supplied or defined.

27. **Raja Yoga — entry 247:** No Digbala assessment is supplied, preventing verification of the required two, three or four qualifying planets.

28. **Raja Yoga — entry 252:** Jupiter and Venus are debilitated in D1, but exalted Navamsa status for either is not established by the supplied D9 data.

29. **Raja Yoga — entry 254:** Saturn occupies a kendra and receives tenth lord Mars’s aspect. Required exaltation or Moolathrikona status is not supplied; own-house status is not substituted for it.

30. **Khalwata Yoga — entry 295:** Cancer Lagna receives Saturn’s aspect, but the required malefic-sign classification is not defined.

31. **Rajabhrashta Yoga — entry 297:** Aroodha Lagna and Aroodha Dwadasa, and therefore their lords’ relationship, are unavailable.

## 4. D. AUDIT SUMMARY

- **Source entries examined:** 300
- **VERIFIED:** 14
- **CONDITIONAL:** 16
- **UNVERIFIABLE:** 31

Repeated names remain separate source entries. Multiple qualifying matches within one entry count only once. No Dasha, transit, event prediction or general life assessment has been added.

## Check Credit Status
https://platform.openai.com/settings/organization/billing/credit-grants

https://platform.openai.com/usage


#ChronoTantra | ChronoYantra | ChronoMantra | ChronoSutra
 [More Information ](http://bit.ly/chronobooks) <br>

[![More Information](https://raw.githubusercontent.com/prithwis/parashar21/refs/heads/main/images/CTYMS-Colab-Footer.png)](https://chronos.yantrajaal.com)